In [0]:
%run ../00-common/01-env-variables


In [0]:
%run ../00-common/03-silver-helpers

In [0]:
dbutils.widgets.text('batch_id',"")
v_batch_id = dbutils.widgets.get('batch_id')

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as F
circuits_df =(
spark.table(bronze_table)
.filter(F.col('batch_id')==v_batch_id)
)
display(circuits_df)

In [0]:
selected_circuits_df =circuits_df.select('circuitId' ,
                                         'circuitName' ,
                                          'lat',
                                           'lng' ,
                                           'locality', 
                                           'country',
                                           'ingestion_timestamp',
                                           'source_file',
                                           'batch_id')


In [0]:
display(selected_circuits_df)

In [0]:
renamed_circuits_df = selected_circuits_df.withColumnsRenamed(
    {"circuitId": "circuit_id",
   "circuitRef": "circuit_ref",
   "name": "circuit_name",
   "lat": "latitude",
   "lng": "longitude"
})

In [0]:
display(renamed_circuits_df)

In [0]:
from pyspark.sql import functions as F
non_null_circuts_df = renamed_circuits_df.filter(F.col('circuit_id').isNotNull())

In [0]:
display(non_null_circuts_df)

In [0]:
distinct_circuits = non_null_circuts_df.dropDuplicates(['circuit_id'])

In [0]:
display(distinct_circuits)

In [0]:
from pyspark.sql import functions as F

circuits_final_df = (
    distinct_circuits.withColumn('circuitName', F.initcap(F.col('circuitName')))
    .withColumn('locality', F.initcap(F.col("locality")))    
)

In [0]:
display(circuits_final_df)

In [0]:
# circuits_final_df = (
#      circuits_final_df
#      .withColumn('created_timestamp' , F.current_timestamp())
#      .withColumn('updated_timestamp' , F.current_timestamp())
# )

In [0]:
# from delta.tables import DeltaTable

# if not spark.catalog.tableExists(silver_table):
#     (
#     circuits_final_df.write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(silver_table)
#     )

# else :
#     delta_table = DeltaTable.forName(spark,silver_table)

# (
#     delta_table.alias("t")
#     .merge(
#         circuits_final_df.alias("s"),
#         "t.circuit_id = s.circuit_id"
#     )
#     .whenMatchedUpdate(
#         condition="s.batch_id >= t.batch_id",
#         set = {
#             "circuitName": "s.circuitName",
#             "latitude": "s.latitude",
#             "longitude": "s.longitude",
#             "locality": "s.locality",
#             "country": "s.country",
#             "ingestion_timestamp": "s.ingestion_timestamp",
#             "source_file": "s.source_file",
#             "batch_id": "s.batch_id",
#             "updated_timestamp": "s.updated_timestamp"
#         }
#     )
#     .whenNotMatchedInsertAll()
#     .execute()
# )

In [0]:
write_to_silver(
    circuits_final_df,
    silver_table,
    "t.circuit_id = s.circuit_id",
    [
        'circuitName',
        'latitude',
        'longitude',
        'locality',
        'country',
        'ingestion_timestamp',
        'source_file',
        'batch_id',
    ],
    v_batch_id
)

In [0]:
display(spark.table(silver_table))
